In [ ]:
#| default_exp augmentations

# Augmentations

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, torch.nn.functional as F
from lightning.pytorch.callbacks import Callback
from torch.distributions.beta import Beta


## Callbacks

In [ ]:
#| export
class MixupCallback(Callback):
    """
    Mixup for 1D data (e.g., time-series).

    This callback applies Mixup to the training data, blending both the input data and the labels.

    See tsai implementation here: https://github.com/timeseriesAI/tsai/blob/bdff96cc8c4c8ea55bc20d7cffd6a72e402f4cb2/tsai/data/mixed_augmentation.py#L43

    Note that this creates non-integer labels/soft labels. Loss functions should be able to handle this.
    """
    def __init__(self, 
                 num_classes,
                 mixup_alpha=0.4, # alpha parameter for the beta distribution
                 return_sequence_padding_mask=True, # whether to return the sequence padding mask
                 ignore_index=-100 # ignore index
                 ):
        super().__init__()
        self.distrib = Beta(mixup_alpha, mixup_alpha)
        self.mixup_alpha = mixup_alpha
        self.return_sequence_padding_mask = return_sequence_padding_mask
        self.ignore_index = ignore_index
        self.num_classes = num_classes

    def on_train_batch_start(self, trainer, pl_module, batch, batch_idx):
        if self.return_sequence_padding_mask:
            x, y, padding_mask = batch  # x: [batch_size, channels, time_steps], y: [batch_size, time_steps]
        else:
            x, y = batch  # x: [batch_size, channels, time_steps], y: [batch_size, time_steps]

        # Sample lambda from a beta distribution
        if batch_idx == 0:
            print("Mixup is being applied!")
        lam = self.distrib.sample((x.size(0), )).to(x.device) # [bs]
        # our mixing coefficient is always ≥ 0.5
        #lam = torch.max(lam, 1 - lam)
        lam = lam.view(-1, 1, 1)  # for input shape [bs, channels, seq_len]

        # Shuffle the batch
        indices = torch.randperm(x.size(0), device=x.device)
        x_shuffled = x[indices]
        y_shuffled = y[indices]

        # create ignore masks
        ignore_mask = (y == self.ignore_index)
        ignore_mask_shuffled = (y_shuffled == self.ignore_index)
        combined_ignore_mask = torch.logical_or(ignore_mask, ignore_mask_shuffled)

        y_clean = torch.where(ignore_mask, torch.zeros_like(y), y)
        y_shuffled_clean = torch.where(ignore_mask_shuffled, torch.zeros_like(y_shuffled), y_shuffled)
        # Create one-hot encodings
        y_onehot = F.one_hot(y_clean.long(), num_classes=self.num_classes).float()
        y_shuffled_onehot = F.one_hot(y_shuffled_clean.long(), num_classes=self.num_classes).float()
        # Zero out the one-hot vectors for ignored indices
        y_onehot = torch.where(ignore_mask.unsqueeze(-1), torch.zeros_like(y_onehot), y_onehot)
        y_shuffled_onehot = torch.where(ignore_mask_shuffled.unsqueeze(-1), torch.zeros_like(y_shuffled_onehot), y_shuffled_onehot)

        # Mixup the inputs and labels
        x_mixed = torch.lerp(x_shuffled, x, lam) # x = lam * x + (1 - lam) * x_shuffled
        y_mixed = torch.lerp(y_shuffled_onehot, y_onehot, lam) # y = lam * y + (1 - lam) * y_shuffled

        # finally assign all probabilities with a -100 mixing to 0
        y_mixed = torch.where(combined_ignore_mask.unsqueeze(-1), torch.zeros_like(y_mixed), y_mixed)
        # # add back in the ignore index where it was mixed with other labels
        # combined_ignore_mask = torch.logical_or(ignore_mask, ignore_mask_shuffled)
        # if combined_ignore_mask.any():
        #     # y_mixed = torch.where(combined_ignore_mask, 
        #     #                         torch.tensor(self.ignore_index, dtype=y_mixed.dtype, device=y_mixed.device), 
        #     #                     y_mixed)
        #     y_mixed = torch.where(
        #         combined_ignore_mask.unsqueeze(-1),
        #         torch.zeros_like(y_mixed),
        #         y_mixed
        #     )
        y_mixed = y_mixed.permute(0,2,1) # [bs x n_classes x n_patches]
        if self.return_sequence_padding_mask:
            # Mix the padding masks using OR operation (1 means padded)
            padding_mask_shuffled = padding_mask[indices]
            # If either mask indicates padding (1), the result should be padded
            padding_mask = torch.logical_or(padding_mask, padding_mask_shuffled)
            batch[0], batch[1], batch[2] = x_mixed, y_mixed, padding_mask
            #return (x_mixed, y_mixed, padding_mask)
        else:
            batch[0], batch[1] = x_mixed, y_mixed
            #return (x_mixed, y_mixed)
        #return batch

In [ ]:
#| notest
x = torch.randn(4,7,90)
x_c = x.clone()
y_og = torch.randint(0, 5, size=(4,90//30))
y_og[1,2] = -100
y_og[2,1] = -100
y_c = y_og.clone()
xxt = MixupCallback(num_classes=5, mixup_alpha=0.4, return_sequence_padding_mask=False)
batch = (x,y_og)
batch = xxt.on_train_batch_start(None, None, batch, 0)
torch.equal(x_c, batch[0]) == False, torch.equal(y_c, batch[1]) == False

Mixup is being applied!


(True, True)

In [ ]:
#| notest
x = torch.randn(4,7,90)
x_c = x.clone()
y_og = torch.randint(0, 2, size=(4,1))
#y_og[1,2] = -100
#y_og[2,1] = -100
y_c = y_og.clone()
xxt = MixupCallback(num_classes=2, mixup_alpha=0.4, return_sequence_padding_mask=False, ignore_index=-100)
batch = (x,y_og)
batch = xxt.on_train_batch_start(None, None, batch, 0)
torch.equal(x_c, batch[0]) == False, torch.equal(y_c, batch[1]) == False

Mixup is being applied!


(True, True)

In [ ]:
#| export
class VariableChannelInput(Callback):
    """
    Randomly adds 0 channels at correct index to the input data to match the number of channels expected by the self supervised model.
    """
    def __init__(self, indexes_to_add_channels, n_channels_expected, channel_dim=1, return_sequence_padding_mask=False):
        super().__init__()
        if not isinstance(indexes_to_add_channels, list) and not isinstance(indexes_to_add_channels, tuple):
            raise ValueError("indexes_to_add_channels must be a list or tuple")
        if not all(isinstance(i, int) for i in indexes_to_add_channels):
            raise ValueError("indexes_to_add_channels must be a list or tuple of integers")
        self.indexes_to_add_channels = indexes_to_add_channels
        self.n_channels_expected = n_channels_expected
        self.return_sequence_padding_mask = return_sequence_padding_mask
        self.channel_dim = channel_dim
    
    def _modify_batch(self, x):
        assert x.shape[self.channel_dim] + len(self.indexes_to_add_channels) == self.n_channels_expected, f"The number of channels expected ({self.n_channels_expected}) by the model must match the number of channels in the input data ({x.shape[self.channel_dim]}) plus the number of channels to add ({len(self.indexes_to_add_channels)})"
        # append zeros to the channels at the end of the input data
        x = torch.cat([x, torch.zeros(x.shape[0], len(self.indexes_to_add_channels), x.shape[2], device=x.device)], dim=self.channel_dim)
        # reindex to match the correct channel order
        new_indices = list(range(x.shape[self.channel_dim]))
        # find the added 0 indices and their new indices
        index_to_add_dict = dict(zip(new_indices[-len(self.indexes_to_add_channels):], self.indexes_to_add_channels))
        for old_idx, new_idx in index_to_add_dict.items():
            new_indices.insert(new_idx, new_indices.pop(old_idx))
        x = torch.index_select(x, dim=self.channel_dim, index=torch.tensor(new_indices, device=x.device))
        return x

    def on_train_batch_start(self, trainer, pl_module, batch, batch_idx):
        if self.return_sequence_padding_mask:
            x, _, _ = batch
        else:
            x, _ = batch
        batch[0] = self._modify_batch(x)
    
    def on_test_batch_start(self, trainer, pl_module, batch, batch_idx):
        if self.return_sequence_padding_mask:
            x, _, _ = batch
        else:
            x, _ = batch
        batch[0] = self._modify_batch(x)
    
    def on_validation_batch_start(self, trainer, pl_module, batch, batch_idx):
        if self.return_sequence_padding_mask:
            x, _, _ = batch
        else:
            x, _ = batch
        batch[0] = self._modify_batch(x)
    
    def on_predict_batch_start(self, trainer, pl_module, batch, batch_idx):
        if self.return_sequence_padding_mask:
            x, _, _ = batch
        else:
            x, _ = batch
        batch[0] = self._modify_batch(x)

In [ ]:
#| notest
cb = VariableChannelInput(indexes_to_add_channels=[0], n_channels_expected=5, channel_dim=1, return_sequence_padding_mask=False)
x = torch.randn(4,4,90)
y = torch.randint(0, 5, size=(4,90//30))
batch = (x,y)
x_new,_ = cb.on_train_batch_start(None, None, batch, 0)
print(x_new.sum(dim=2))

torch.Size([4, 4, 90])
torch.Size([4, 5, 90])
tensor([[  0.0000,  -1.8939,   1.0597,   0.1701,   1.4365],
        [  0.0000,   2.0693,  12.3198,  16.0288,   8.4201],
        [  0.0000,   5.4088, -13.7100,   1.0756,   2.7868],
        [  0.0000,  12.0397,  -4.6842, -10.2520,  -3.4211]])


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()